# Tarea 2: Fundamentos de Python
## Ciencia de Datos Ambientales - UTEC

**Nombre:** Camila Luana Guevara Dávila  
**Puntaje total:** 20 puntos

**Instrucciones:**
- Completa todos los problemas en este notebook
- Escribe tu codigo en las celdas proporcionadas
- Ejecuta todas las celdas antes de entregar
- Sube el archivo `.ipynb` completado al modulo correspondiente en Canvas

**Integridad academica:** Tarea individual. Puedes consultar materiales del curso y documentacion de Python, pero todo el codigo debe ser tuyo.

---

## Problema 1: Procesador de Nombres de Archivos Landsat (10 puntos)

Trabajas con imagenes satelitales Landsat del Peru. Los nombres siguen el formato:

```
LC08_L2SP_008067_20240615_02_T1_SR_B4.TIF
```
Componentes: `{sensor}_{nivel}_{path_row}_{fecha}_{coleccion}_{tier}_SR_{banda}.TIF`

> Los paths 003-009, filas 062-071 cubren el territorio peruano (Madre de Dios, Loreto, Lima, Cusco).

### Tus tareas:

**Parte A (4 pts):** Funcion `procesar_nombre_landsat(nombre_archivo)` que devuelva un diccionario con:
- `sensor` (ej. "LC08"), `path` (ej. "008"), `row` (ej. "067")
- `fecha` formateada como "AAAA-MM-DD"
- `banda` (ej. "B4")

**Parte B (3 pts):** Funcion `clasificar_banda(banda)` que devuelva el nombre segun la tabla:

| Banda | Nombre |
|-------|--------|
| B1 | Aerosol costero | B2 | Azul | B3 | Verde | B4 | Rojo |
| B5 | Infrarrojo cercano (NIR) | B6 | SWIR1 | B7 | SWIR2 |

Si no esta en la tabla, devuelve "Desconocida".

**Parte C (3 pts):** Procesa la lista de archivos: parsea, imprime resumen (fecha/path/row/banda) y cuenta cuantas fechas unicas hay.

In [4]:
# Archivos Landsat sobre el Peru (paths 008-009: Madre de Dios, Ucayali, Loreto)
archivos = [
    "LC08_L2SP_008067_20240615_02_T1_SR_B4.TIF",
    "LC08_L2SP_008067_20240615_02_T1_SR_B5.TIF",
    "LC08_L2SP_008067_20240701_02_T1_SR_B3.TIF",
    "LC08_L2SP_009067_20240615_02_T1_SR_B4.TIF",
    "LC09_L2SP_008067_20240708_02_T1_SR_B6.TIF",
    "LC08_L2SP_008067_20240701_02_T1_SR_B4.TIF",
]

# Parte A: funcion procesar_nombre_landsat
def procesar_nombre_landsat(nombre_archivo):
  partes = nombre_archivo.split("_")
  sensor = partes[0]
  path_row = partes[2]
  path = path_row[0:3]
  row = path_row[3:6]
  fecha_original = partes[3]
  fecha = fecha_original[0:4] + "-" + fecha_original[4:6] + "-" + fecha_original[6:8]
  banda = partes[7].replace(".TIF", "")
  resultado = {"sensor": sensor, "path": path, "row": row, "fecha": fecha, "banda": banda}
  return resultado

for i in archivos:
  resultado=procesar_nombre_landsat(i)
  print(resultado)

# Parte B: funcion clasificar_banda
print('-------------------------------------')
print('Clasificación de bandas:')
def clasificar_banda(banda):
  if banda == "B1":
    return "Aerosol costero"
  elif banda == "B5":
    return "Infrarrojo cercano (NIR)"
  else:
    return "Desconocida"
for i in archivos:
  resultado = procesar_nombre_landsat(i)
  banda=resultado["banda"]
  banda_resultado=clasificar_banda(banda)
  print(banda_resultado)

# Parte C: procesar todos los archivos
fechas_unicas = []
print('-------------------------------------')
print('Resumen:')
for i in archivos:
    resultado = procesar_nombre_landsat(i)
    print("Fecha:", resultado ["fecha"], "Path:", resultado ["path"],"Row:", resultado ["row"],"Banda:", resultado ["banda"])
    if resultado ['fecha']  not in fechas_unicas:
      fechas_unicas.append(resultado ['fecha'])

print("Fechas únicas:", len(fechas_unicas))

{'sensor': 'LC08', 'path': '008', 'row': '067', 'fecha': '2024-06-15', 'banda': 'B4'}
{'sensor': 'LC08', 'path': '008', 'row': '067', 'fecha': '2024-06-15', 'banda': 'B5'}
{'sensor': 'LC08', 'path': '008', 'row': '067', 'fecha': '2024-07-01', 'banda': 'B3'}
{'sensor': 'LC08', 'path': '009', 'row': '067', 'fecha': '2024-06-15', 'banda': 'B4'}
{'sensor': 'LC09', 'path': '008', 'row': '067', 'fecha': '2024-07-08', 'banda': 'B6'}
{'sensor': 'LC08', 'path': '008', 'row': '067', 'fecha': '2024-07-01', 'banda': 'B4'}
-------------------------------------
Clasificación de bandas:
Desconocida
Infrarrojo cercano (NIR)
Desconocida
Desconocida
Desconocida
Desconocida
-------------------------------------
Resumen:
Fecha: 2024-06-15 Path: 008 Row: 067 Banda: B4
Fecha: 2024-06-15 Path: 008 Row: 067 Banda: B5
Fecha: 2024-07-01 Path: 008 Row: 067 Banda: B3
Fecha: 2024-06-15 Path: 009 Row: 067 Banda: B4
Fecha: 2024-07-08 Path: 008 Row: 067 Banda: B6
Fecha: 2024-07-01 Path: 008 Row: 067 Banda: B4
Fechas 

---
## Problema 2: Inventario Forestal en Madre de Dios (10 puntos)

El **SERFOR** realiza inventarios forestales en Madre de Dios. Los datos incluyen valores faltantes (`-999`) y mediciones con posibles errores.

**Parte A (3 pts):** Funcion `calcular_area_basal(dap_cm)`:
- Devuelve AB en m2: $AB = \pi 	\times  (DAP/200)^2$
- Devuelve `None` si DAP <= 0 o == -999

**Parte B (3 pts):** Funcion `clasificar_arbol(dap_cm, altura_m)` que devuelva:
- `clase`: "Brinzal" (<10cm), "Latizal" (10-25cm), "Fustal menor" (25-50cm), "Fustal mayor" (>=50cm)
- `alerta`: True si DAP > 200cm, altura > 60m, o altura < 1m con DAP > 10cm

**Parte C (4 pts):** Procesa los datos:
1. Para cada árbol, calcule el área basal y clasifíquelo.
2. Omita los árboles con datos faltantes (valores -999).
3. Imprima una advertencia para los árboles marcados.
4. Calcule e imprima las estadísticas descriptivas:
- Número total de árboles válidos
- Área basal total (suma de todos los árboles válidos)
- Cantidad de árboles en cada clase de tamaño
- Número de registros marcados

In [6]:
# Inventario forestal - Madre de Dios, Peru (datos SERFOR)
# Formato: [id, especie, dap_cm, altura_m]
datos_arboles = [
    [1,  "Swietenia macrophylla",      35.4, 22.1],   # Caoba
    [2,  "Cedrela odorata",            28.2, 18.5],   # Cedro
    [3,  "Cedrelinga cateniformis",   -999,  25.0],   # Tornillo - DAP faltante
    [4,  "Virola surinamensis",        18.7, 12.3],   # Cumala
    [5,  "Dipteryx micrantha",         52.1, 24.8],   # Shihuahuaco
    [6,  "Calycophyllum spruceanum",    8.5,  6.2],   # Capirona
    [7,  "Terminalia oblonga",         45.0, 85.0],   # Yacushapana - altura sospechosa
    [8,  "Cedrelinga cateniformis",    62.3, 28.4],   # Tornillo
    [9,  "Swietenia macrophylla",      41.2, -999],   # Caoba - altura faltante
    [10, "Hura crepitans",             22.5,  0.5],   # Catahua - sospechoso
    [11, "Schizolobium parahybum",      5.2,  3.1],   # Pino chuncho
    [12, "Guazuma crinita",            38.9, 21.7],   # Bolaina
]

# Parte A: Escribe la función calcular_area_basal aqui:
def calcular_area_basal(dap_cm):
  if dap_cm <= 0 or dap_cm == -999:
    return None
  else:
    area_basal = round(3.14 * (dap_cm / 200) ** 2,4)
    return area_basal

# Parte B: Escribe la función clasificar_arbol aquí:
def clasificar_arbol(dap_cm, altura_m):
  if dap_cm < 10:
    clase = "Brinzal"
  elif dap_cm < 25:
    clase = "Latizal"
  elif dap_cm < 50:
    clase = "Fustal menor"
  else:
    clase = "Fustal mayor"

  if dap_cm > 200 or altura_m > 60 or (altura_m < 1 and dap_cm > 10):
    alerta = True
  else:
    alerta = False
  resultado = {"clase": clase, "alerta": alerta}
  return resultado

# Parte C: Procesar los datos e imprimir los resultados
arbol_valido = 0
AB = 0
Brinzal = 0
Latizal = 0
Fustalmenor = 0
Fustalmayor = 0
marcado = 0

for i in datos_arboles:
  dap_cm = i[2]
  altura_m = i[3]
  if dap_cm == -999 or altura_m == -999:
    continue
  arbol_valido = arbol_valido + 1
  area_basal = calcular_area_basal(dap_cm)
  AB = AB + area_basal
  clasificar = clasificar_arbol(dap_cm, altura_m)
  if clasificar['clase']== "Brinzal":
      Brinzal= Brinzal + 1
  elif clasificar['clase']=='Latizal':
      Latizal= Latizal + 1
  elif clasificar['clase']=='Fustal menor':
      Fustalmenor= Fustalmenor + 1
  elif clasificar['clase']=='Fustal mayor':
      Fustalmayor= Fustalmayor + 1
  print("ID:", i[0],"|","Especie:", i[1],"|", "AB:", area_basal, "|","Clasificación:", clasificar['clase'])

  if clasificar["alerta"] == True:
    marcado=marcado+1
    print("Advertencia: árbol", i[0], "marcado")

print("-----------------------------")
print("Estadísticas descriptivas:")
print("Número total de árboles válidos:", arbol_valido)
print("Área basal total (suma de todos los árboles válidos):", round(AB,4))
print("Cantidad de árboles en cada clase de tamaño:")
print("Brinzal:", Brinzal)
print("Latizal:", Latizal)
print("Fustal menor:", Fustalmenor)
print("Fustal mayor:", Fustalmayor)
print("Número de registros marcados:", marcado)

ID: 1 | Especie: Swietenia macrophylla | AB: 0.0984 | Clasificación: Fustal menor
ID: 2 | Especie: Cedrela odorata | AB: 0.0624 | Clasificación: Fustal menor
ID: 4 | Especie: Virola surinamensis | AB: 0.0275 | Clasificación: Latizal
ID: 5 | Especie: Dipteryx micrantha | AB: 0.2131 | Clasificación: Fustal mayor
ID: 6 | Especie: Calycophyllum spruceanum | AB: 0.0057 | Clasificación: Brinzal
ID: 7 | Especie: Terminalia oblonga | AB: 0.159 | Clasificación: Fustal menor
Advertencia: árbol 7 marcado
ID: 8 | Especie: Cedrelinga cateniformis | AB: 0.3047 | Clasificación: Fustal mayor
ID: 10 | Especie: Hura crepitans | AB: 0.0397 | Clasificación: Latizal
Advertencia: árbol 10 marcado
ID: 11 | Especie: Schizolobium parahybum | AB: 0.0021 | Clasificación: Brinzal
ID: 12 | Especie: Guazuma crinita | AB: 0.1188 | Clasificación: Fustal menor
-----------------------------
Estadísticas descriptivas:
Número total de árboles válidos: 10
Área basal total (suma de todos los árboles válidos): 1.0314
Cantid

---
## Lista de verificacion
- [ ] Todas las celdas corren sin errores
- [ ] Ambos problemas estan completos
- [ ] Salidas visibles en todas las celdas
- [ ] Nombre incluido